In [ ]:
import pandas as pd

from preparar_base_pede import carregar_base_unificada

df = carregar_base_unificada()

df.head()

In [ ]:
df.info()

In [ ]:
df['risco'] = df['Pedra Atual'].isin(['Quartzo', 'ÃƒÂgata']).astype(int)
df['risco'].value_counts()

In [ ]:
df['Pedra Atual'].value_counts()

In [ ]:
features_modelo = [
    'IDA', 'IEG', 'IPV', 'Matem', 'Portug', 'Inglês', 'IAA', 'IPS', 'IAN', 'Defas'
]

df_model = df[features_modelo + ['risco']].copy()

In [ ]:
for coluna in features_modelo:
    df_model[coluna] = pd.to_numeric(df_model[coluna], errors='coerce')

df_model = df_model.dropna(subset=features_modelo + ['risco'])

In [ ]:
X = df_model[features_modelo]
y = df_model['risco']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)  # X_train precisa ser DataFrame

In [ ]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)
print(classification_report(y_test, pred))

In [ ]:
import pandas as pd
import numpy as np

importances = model.feature_importances_
features = X.columns

feat_imp = pd.DataFrame({
    'variavel': features,
    'importancia': importances
}).sort_values(by='importancia', ascending=False)

feat_imp.head(15)

In [ ]:
import matplotlib.pyplot as plt

top = feat_imp.head(10)

plt.figure(figsize=(8,5))
plt.barh(top['variavel'][::-1], top['importancia'][::-1])
plt.title("Principais fatores que levam ao risco educacional")
plt.show()

In [ ]:
proba = model.predict_proba(X_test)[:,1]

In [ ]:
import pandas as pd
resultado = pd.DataFrame({
    'real': y_test,
    'prob_risco': proba
})
resultado.head(10)

In [ ]:
def classificar_risco(p):
    if p < 0.30:
        return "Baixo"
    elif p < 0.60:
        return "Moderado"
    elif p < 0.80:
        return "Alto"
    else:
        return "Crítico"

resultado["nivel_risco"] = resultado["prob_risco"].apply(classificar_risco)
resultado.head(10)

In [ ]:
import joblib
joblib.dump(model, "modelo_risco.pkl")

## AnÃƒÂ¡lises para responder ao PDF

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df_analise = df.copy()
df_analise["nivel_risco_real"] = df_analise["risco"].map({0: "Sem risco", 1: "Em risco"})

indicadores = [
    "IAA", "IDA", "IEG", "IPV", "IPS", "IAN", "Matem", "Portug", "Inglês", "Defas"
]

for coluna in indicadores:
    if coluna in df_analise.columns:
        df_analise[coluna] = pd.to_numeric(df_analise[coluna], errors="coerce")

df_analise[indicadores].describe().T

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    df_analise["IAA"],
    df_analise["IDA"],
    c=df_analise["IEG"],
    cmap="viridis",
    alpha=0.6
)
plt.colorbar(label="IEG")
plt.xlabel("IAA")
plt.ylabel("IDA")
plt.title("IAA vs IDA, colorido pelo IEG")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    df_analise["IAA"],
    df_analise["IEG"],
    c=df_analise["risco"],
    cmap="coolwarm",
    alpha=0.6
)
plt.colorbar(label="Risco real")
plt.xlabel("IAA")
plt.ylabel("IEG")
plt.title("IAA vs IEG, colorido pelo risco real")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
corr_cols = [
    "IAA", "IDA", "IEG", "IPV", "IPS", "IAN", "Matem", "Portug", "Inglês", "Defas"
]
corr = df_analise[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
ax.set_title("CorrelaÃƒÂ§ÃƒÂ£o entre indicadores")

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
comparacao_ano = (
    df_analise.groupby("Ano Referencia")[["IAA", "IDA", "IEG", "IPV"]]
    .mean()
    .round(2)
)

comparacao_ano

In [ ]:
comparacao_ano.plot(kind="bar", figsize=(10, 6))
plt.title("ComparaÃƒÂ§ÃƒÂ£o mÃƒÂ©dia dos indicadores por ano")
plt.ylabel("MÃƒÂ©dia")
plt.xlabel("Ano Referencia")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
risco_por_ano = pd.crosstab(df_analise["Ano Referencia"], df_analise["nivel_risco_real"], normalize="index").round(3)
risco_por_ano

In [ ]:
risco_por_ano.plot(kind="bar", stacked=True, figsize=(10, 6), color=["#7bb661", "#d95f5f"])
plt.title("DistribuiÃƒÂ§ÃƒÂ£o do risco real por ano")
plt.ylabel("ProporÃƒÂ§ÃƒÂ£o")
plt.xlabel("Ano Referencia")
plt.xticks(rotation=0)
plt.legend(title="SituaÃƒÂ§ÃƒÂ£o")
plt.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
fase_analise = df_analise["Fase"].astype(str)

risco_por_fase = (
    pd.crosstab(fase_analise, df_analise["nivel_risco_real"], normalize="index")
    .sort_index()
    .round(3)
)

risco_por_fase


In [ ]:
risco_por_fase.plot(kind="bar", stacked=True, figsize=(10, 6), color=["#7bb661", "#d95f5f"])
plt.title("DistribuiÃƒÂ§ÃƒÂ£o do risco real por fase")
plt.ylabel("ProporÃƒÂ§ÃƒÂ£o")
plt.xlabel("Fase")
plt.xticks(rotation=0)
plt.legend(title="SituaÃƒÂ§ÃƒÂ£o")
plt.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
faixas_iaa = pd.cut(
    df_analise["IAA"],
    bins=[0, 4, 6, 8, 10],
    labels=["Baixa", "MÃƒÂ©dia", "Alta", "Muito alta"],
    include_lowest=True
)

coerencia_iaa = (
    df_analise.assign(Faixa_IAA=faixas_iaa)
    .groupby("Faixa_IAA")[["IDA", "IEG", "IPV", "risco"]]
    .mean()
    .round(2)
)

coerencia_iaa